# Fine-tuning YOLO26s — Detecção de Placas

**Passo único obrigatório:** Ative GPU T4 em `Editar → Configurações do notebook → GPU`

**Tempo estimado:** ~30 min (GPU T4)

In [ ]:
# ── Configurações ──────────────────────────────────────────
MODELO_BASE      = "yolo26s"   # yolo26n (rápido) ou yolo26s (preciso)
EPOCHS           = 50
BATCH            = 32          # T4: 32; se der OOM use 16

# Dataset: deixe ROBOFLOW_API_KEY vazio para usar HuggingFace (~800 imgs)
#          ou coloque sua key para usar Roboflow (~24k imgs — muito melhor)
ROBOFLOW_API_KEY = ""          # cole sua key aqui para usar ~24k imagens
# ───────────────────────────────────────────────────────────

In [ ]:
# Instala dependências
!pip install ultralytics datasets roboflow --quiet

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠ NÃO DETECTADA — ative GPU T4")
print("CUDA:", torch.version.cuda)

In [ ]:
# Baixa dataset
from pathlib import Path
BASE = Path("/content/dataset")

if ROBOFLOW_API_KEY:
    # ── Roboflow: ~24.000 imagens (CC BY 4.0) ──────────────
    from roboflow import Roboflow
    print("Baixando dataset Roboflow (~24k imagens)...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
    dataset = project.version(4).download("yolov8", location=str(BASE))
    DATA_YAML = str(BASE / "data.yaml")
    print("Dataset em:", dataset.location)

else:
    # ── HuggingFace: ~800 imagens (sem cadastro) ───────────
    from datasets import load_dataset
    print("Baixando dataset HuggingFace (~800 imagens)...")
    ds = load_dataset(
        "keremberke/license-plate-object-detection",
        name="full",
        trust_remote_code=True,   # necessário para datasets com script customizado
    )
    for split, nome in [("train", "train"), ("validation", "valid"), ("test", "test")]:
        (BASE / nome / "images").mkdir(parents=True, exist_ok=True)
        (BASE / nome / "labels").mkdir(parents=True, exist_ok=True)
        for i, sample in enumerate(ds[split]):
            sample["image"].save(str(BASE / nome / "images" / f"{i:06d}.jpg"))
            W, H = sample["image"].size
            lines = []
            for obj in sample["objects"]:
                x1, y1, x2, y2 = obj["bbox"]
                cx = ((x1+x2)/2)/W; cy = ((y1+y2)/2)/H
                w  = (x2-x1)/W;     h  = (y2-y1)/H
                lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            (BASE / nome / "labels" / f"{i:06d}.txt").write_text("\n".join(lines))
        print(f"  {nome}: {len(ds[split])} imagens")
    (BASE / "data.yaml").write_text(
        f"path: {BASE}\ntrain: train/images\nval: valid/images\n"
        f"test: test/images\nnc: 1\nnames: ['license-plate']\n"
    )
    DATA_YAML = str(BASE / "data.yaml")

print("\nDataset pronto:", DATA_YAML)

In [ ]:
# Fine-tuning
from ultralytics import YOLO
from pathlib import Path

model = YOLO(f"{MODELO_BASE}.pt")  # baixa automaticamente da Ultralytics

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=0,
    name="placas_yolo26s",
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.3,
    shear=2.0, perspective=0.0002,
    flipud=0.0, fliplr=0.0,
    mosaic=0.5, mixup=0.1, copy_paste=0.1,
    patience=20,
    save_period=10,
)

best = Path(results.save_dir) / "weights" / "best.pt"
print("\nMelhor modelo:", best)
print("mAP@50:", results.results_dict.get('metrics/mAP50(B)', 'N/A'))

In [ ]:
# Exporta para ONNX
from ultralytics import YOLO
from pathlib import Path

best = Path(results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best))
onnx_path = model.export(format="onnx", imgsz=640, opset=12)
print("ONNX exportado:", onnx_path)

In [ ]:
# Baixa os modelos para o seu PC
from google.colab import files
from pathlib import Path

best_pt   = Path(results.save_dir) / "weights" / "best.pt"
best_onnx = best_pt.with_suffix(".onnx")

print("Baixando best.pt ...")
files.download(str(best_pt))

print("Baixando best.onnx ...")
files.download(str(best_onnx))

print()
print("Copie o best.onnx para: leitura-placas/models/plate_detector.onnx")
print("e reinicie o servidor.")